# RAG Pipeline Flow 
This notebook walks through every stage of the RAG pipeline in order.
Each cell maps directly to one file in the `indexing/` or `generation/` folder.

**Pipeline:**
```
INDEXING                          GENERATION
────────────────────────────      ──────────────────────────
1. Load     sample_news.json      5. Embed query
2. Chunk    split into pieces     6. Retrieve  vector search
3. Embed    text → vectors        7. Generate  LLM + context
4. Store    vectors → Qdrant      8. Response  citations
```

In [2]:
import json, os 
from dotenv import load_dotenv
from datetime import datetime # parsing date time

load_dotenv()  # reads from .env

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Add OPENAI_API_KEY to your .env file"
print("✓ API key loaded")

✓ API key loaded


# Indexing Pipeline

## Stage 1: Load documents
`indexing/loaders/file_loader.py`

Read raw documents and outputs a list of `documents`

In [3]:
class RawDocument:
    def __init__(self, id, title, text, url, publish_date, category="", tags=None):
        self.id = id
        self.title = title
        self.text = text
        self.url = url
        self.publish_date = publish_date
        self.category = category
        self.tags = tags if tags is not None else []
        
def load(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
        
    docs = []
    for doc in raw: 
        docs.append(RawDocument(
            id=doc["id"],
            title=doc["title"],
            text=doc["content"],
            url=doc["url"],
            publish_date=datetime.fromisoformat(doc["publish_date"]),
            category=doc.get("category", ""),
            tags=doc.get("tags", []),
        ))
        
    return docs
        
docs = load("./data/news.json")

print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(f" [{doc.id}] {doc.title}")

Loaded 11 documents

 [20260203_1] 樂團「微醺開根」首發專輯 唱當代族群故事
 [20260203_2] 排灣童聲唱出部落記憶 春日國小推《春迴》專輯
 [20260203_3] 仁愛鄉梅子白蘭地風味絕佳 奪國際一星獎章
 [20260203_4] 復興推「部落社區再復興」 凝共識促永續發展
 [20260203_5] 陳義信用行動感謝 職棒生涯文物捐贈退輔會
 [20260203_6] 台北國際電玩展 設計商「赴原Saikin」展新作
 [20260203_7] 嘉義文化科技創意賽頒獎 近50組團隊參賽
 [20260203_8] 《村裡遇見熊》書展亮相 記錄人熊共生故事錄
 [20260203_9] 台北國際書展登場 原民館推講座、交流活動
 [20260203_11] 首批承領人正式取得 大南澳土地所有權狀
 [20260203_12] 冒用縣府名義詐捐 花蓮縣府急澄清籲防詐


# Stage 2: Chunk
`indexing/chunkers/semantic_chunker.py`

Split each document into smaller pieces. 

**Why**: LLMs have content limits. Smaller chunks = more precise retrieval

Key rule: every chunk must keep `title`, `url`, `publish_date` for citation later.

In [39]:
import re

class Chunk:
    def __init__(self, chunk_id, source_id, text, title, url, publish_date):
        self.chunk_id = chunk_id
        self.source_id = source_id
        self.text = text
        self.title = title
        self.url = url
        self.publish_date = publish_date

def chunk(doc: RawDocument, max_sentences=3, overlap_sentences=1):
    """
    Sentence boundaries in Chinese naturally align with meaning boundaries. 
    Approach: 
        Chunk by sentences
    Method:
        Slide a window of max_sentences over the sentence list.
        Each step moves forward by (max_sentences - overlap_sentences),
        so the last overlap_sentences cards are always carried into the next chunk.
    """
    # Split on Chinese punctuation
    # sentences = re.split(r"(?<=[。！？])", doc.text)
    sentences = re.findall(r".+?[。！？]」?|.+?」", doc.text, re.DOTALL)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    i = 0
    step = max_sentences - overlap_sentences   # how far to advance each time
    while i < len(sentences):
        window = sentences[i : i + max_sentences]  # take up to max_sentences cards
        chunks.append("".join(window))
        i += step

    return [
        Chunk(
            source_id=doc.id,
            chunk_id=f"{doc.id}_{i}",
            text=text,
            title=doc.title,
            url=doc.url,
            publish_date=doc.publish_date,
        )
        for i, text in enumerate(chunks)
    ]

all_chunks = []
for doc in docs:
    all_chunks.extend(chunk(doc))

print(f"{len(docs)} documents → {len(all_chunks)} chunks\n")

11 documents → 52 chunks



In [41]:
# Inspect: sentences → chunks for a given news_id
news_id = "20260203_3"

doc = next(d for d in docs if d.id == news_id)

# Step 1: show how the article splits into sentences
sentences = re.findall(r".+?[。！？]」?|.+?」", doc.text, re.DOTALL)
sentences = [s.strip() for s in sentences if s.strip()]

print(f"=== Sentences ({len(sentences)} total) ===")
for i, s in enumerate(sentences):
    print(f"  [S{i}] ({len(s)} chars) {s}")

# Step 2: show how sentences are grouped into chunks
print(f"\n=== Chunks (max_sentences=3, overlap=1) ===")
example_chunks = [c for c in all_chunks if c.source_id == news_id]
for c in example_chunks:
    print(f"\n  [{c.chunk_id}] ({len(c.text)} chars)")
    print(f"  {c.text}")

=== Sentences (6 total) ===
  [S0] (111 chars) 南投縣仁愛鄉原住民果樹生產合作社，自108年成立以來，就不斷創新研發仁愛三酸的各種加工食品，不但自有小型的加工廠，還與專業廠商合作，以梅子、檸檬以及楊梅等作物，研發各種酒類產品，在全國百大以及南投十大伴手禮中，都榜上有名。
  [S1] (116 chars) 仁愛鄉原住民果樹生產合作社理事長 鄭春木：「成立之後我們最主要是要提高，農民的收益收入，因為我們這裡是三酸，有檸檬、梅子、楊梅，我們必須要研發更多的產品，來提高我們的行銷，縣府的十大伴手禮，然後我們全國的百大伴手禮，每一年都得獎。」
  [S2] (106 chars) 雖然不少研發的加工食品，在國內頗受好評，但合作社並未因此停下腳步，還嘗試將新研發的白蘭地梅酒，送往比利時參加ITI國際風味評鑑所競賽，並在來自全球的專業主廚，以及侍酒師的評審團嚴格評選下，榮獲風味絕佳的星等獎章。
  [S3] (128 chars) 仁愛鄉原住民果樹生產合作社經理 林明煌：「有一次就是看到人家用那個白蘭地，那個用橡木桶把它做出一個白蘭地來，所以我們想說看看是不是可以，用我們的梅子酒做成我們的白蘭地，我們就去比賽比比看，那去年呢參加評鑑，那今年評鑑結果出爐了，所以我們得到了一星的獎章。」
  [S4] (70 chars) 仁愛鄉原住民果樹生產合作社理事長 鄭春木：「所以我們這一次參加比利時，得到一星獎，得獎之後我們的價錢就比較提高一點，所以我們也比較好行銷。」
  [S5] (103 chars) 梅子檸檬以及楊梅，是仁愛鄉內重要經濟作物之一，合作社藉由產官學合作不斷創新研發產品，不僅提升仁愛鄉農特產知名度，讓農民收益能夠更好，這次能在國外獲獎，也成功在國際舞臺上，讓更多人看見台灣原鄉農產加工的實力。

=== Chunks (max_sentences=3, overlap=1) ===

  [20260203_3_0] (333 chars)
  南投縣仁愛鄉原住民果樹生產合作社，自108年成立以來，就不斷創新研發仁愛三酸的各種加工食品，不但自有小型的加工廠，還與專業廠商合作，以梅子、檸檬以及楊梅等作物，研發各種酒類產品，在全國百大以及南投十大伴手禮中，都榜上有名。仁愛鄉原住民果樹生產合作社理事長 鄭春木：「成立之後我們最主要

# Stage 3: Embed
`indexing/embedders/bge_m3.py`

Convert each chunk's text into a vector representation. *Why*: vectors capture semantic meaning and similar meaning means similar vector.

In [ ]:
from sentence_transformers import SentenceTransformer

# Embedder Choice: "BAAI/bge-m3" / ""
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

texts = [chunk.text for chunk in all_chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"\nEmbedded {len(embeddings)} chunks")
print(f"Vector dimension: {len(embeddings[0])}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4153.60it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00, 43.82it/s]


Embedded 80 chunks
Vector dimension: 384


# Stage 4: Store
`indexing/vector_stores/qdrant_store.py`

Save vectors + chunks metadata into Qdrant.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct 

# :memory: = no Docker needed. Swap to host="localhost" when using Docker
client = QdrantClient(":memory:")
